# Guider Star Ellipticity (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-15  
**Last Modified:** 2026-07-15  
**Status:** In Progress  
**Keywords:** guiders, ellipticity, image motion, star tracking, PSF  

## Description

Kick-off notebook for a study of the ellipticity of stars in the LSSTCam guide sensors.
The guiders run at 5 Hz, producing one *stamp* (a 50 ms integration) per readout. This
notebook loads a sample exposure, runs the `GuiderStarTracker`, and documents the
structure of the resulting per-stamp star catalog so that later notebooks can
characterize ellipticity two ways:

1. **Per-stamp ellipticity** — the shape of the star image within a single 50 ms stamp
   (`e1`, `e2`, `ixx`, `iyy`, `ixy`, `fwhm`).
2. **Image-motion ellipticity** — the effective ellipticity derived from the second
   moments of the star's centroid motion *across* the stamps within an exposure
   (`dx`, `dy` / `dalt`, `daz` vs `elapsed_time`).

Key functionality:
1. Read guider data with `GuiderReader`.
2. Track stars with `GuiderStarTracker` to produce the `stars` catalog.
3. Identify and document every column of the output table, grouped by role.

**Output:** An annotated view of the `GuiderStarTracker` output table (schema, dtypes,
summary statistics, per-detector counts).

**Based on:** `guider/guider_star_detection_demo.ipynb`; `summit_utils` guiders module
(https://github.com/lsst-so/summit_utils/tree/main/python/lsst/summit/utils/guiders).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-15 | Aaron Roodman | Initial version: load data, run tracker, document output table |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Data Access — GuiderReader](#data)
4. [Edge-star recovery](#edge)
5. [Track Stars — GuiderStarTracker](#tracking)
6. [Identify the Output Table](#schema)
7. [Per-Detector Summary](#summary)
8. [Guide-star images (movie & coadd)](#images)
9. [Shape from Second Moments: Optics vs Turbulence](#ellip)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================

dayObs = 20260709          # Observation date (YYYYMMDD)
seqNum = 808               # Sequence number within the night

butler_repo = 'main'       # Butler repository
collections = ['LSSTCam/raw/guider', 'LSSTCam/raw/all']
view = 'dvcs'              # Guider readout orientation view

guider_hz = 5.0            # Guider readout rate (Hz)
stamp_ms = 50.0            # Nominal per-stamp integration (milliseconds)

# Tracker quality cuts
min_snr = 10.0
max_ellipticity = 0.7
edge_margin = 3            # GuiderStarTrackerConfig edgeMargin (pixels)

# Edge-star recovery (see 'Edge-star recovery' section)
recover_edge_stars = True  # admit stars in partial cutouts near the ROI edge
min_finite_fraction = 0.5  # min finite fraction of a cutout when recovering

# Moment measurement (see moment_weighting_analysis.md)
ap_nsigma = 4.0            # aperture radius = ap_nsigma * HSM sigma (per detector)
cen_niter = 3              # iterations for the fixed-Gaussian weighted centroid

# Image display (movie + coadd)
image_cutout = 30          # zoomed ROI cutout size (pixels) for the plots
image_plo = 50             # lower percentile for intensity scaling
image_phi = 98             # upper percentile for intensity scaling

expid = dayObs * 100000 + seqNum
print(f'expid = {expid}')

<a id='setup'></a>
## Setup & Imports

In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Reusable library in guider/code (run this notebook from the guider/ dir).
for _cand in ('code', 'guider/code', os.path.join(os.getcwd(), 'code')):
    if os.path.isdir(_cand):
        sys.path.insert(0, os.path.abspath(_cand))
        break
from guiderMoments import (MomentConfig, decomposeExposure, momentsToDataFrame,
                           centroidPsd)
from guiderEdgeRecovery import (makeTrackerConfig, configSupportsMinFiniteFraction,
                                stackHasRobustDetection)

# summit_utils guiders
from lsst.summit.utils.guiders.reading import GuiderReader
from lsst.summit.utils.guiders.tracking import GuiderStarTracker, GuiderStarTrackerConfig
from lsst.daf.butler import Butler

<a id='data'></a>
## Data Access — GuiderReader

In [ ]:
butler = Butler(butler_repo, collections=collections)

reader = GuiderReader(butler, view=view)
guiderData = reader.get(dayObs=dayObs, seqNum=seqNum, doSubtractMedian=True)

print(f'nStamps      : {len(guiderData)}')
print(f'guiderNames  : {guiderData.guiderNames}')
print(f'detNameMax   : {guiderData.detNameMax}')
print(f'filter       : {guiderData.header.get("filter", "UNKNOWN")}')
print(f'camRotAngle  : {guiderData.camRotAngle:.3f} deg')

<a id='edge'></a>
## Edge-star recovery

Historically some guide sensors reported *No sources detected* / *No stars found* even with a
bright star present, from two `summit_utils.guiders.detection` issues on the median coadd:

- the **absolute** `isBlankImage` cut (peak < 300 ADU &rArr; "blank", detection skipped), and
- `measureStarOnStamp` rejecting any cutout containing a NaN (near-edge stars).

Both are fixed on the **`deploy-summit`** branch (what RubinTV runs): a MAD-based `isBlankImage`
(`peakSnrMin`), a single-stamp fallback, and border-NaN handling. `makeTrackerConfig()` below
detects that and builds a plain `GuiderStarTrackerConfig` with **no patching**. On an older
stack it falls back to setting `minFiniteFraction` (if present) or monkeypatching
`measureStarOnStamp` (`code/guiderEdgeRecovery.py`). A *monkeypatch* = replacing a function on
an already-imported module at runtime, without editing its source.

In [ ]:
# makeTrackerConfig (next cell) adapts to the installed stack; report what it will do.
robust = stackHasRobustDetection()
print('Native robust guider detection (deploy-summit+):', robust)
print('config has minFiniteFraction (interim fix)     :', configSupportsMinFiniteFraction())
if robust:
    print('-> using native detection; no monkeypatch, min_finite_fraction ignored.')
else:
    print('-> recover_edge_stars =', recover_edge_stars,
          '| min_finite_fraction =', min_finite_fraction)

<a id='tracking'></a>
## Track Stars — GuiderStarTracker

Run the tracker with the default config. `refCatalog=None` triggers a
self-generated reference catalog built from the coadd of each guider's stamps.
One star is tracked per guide sensor (the highest-SNR candidate that passes the
quality cuts), measured on every 50 ms stamp.

In [ ]:
config = makeTrackerConfig(
    minFiniteFraction=min_finite_fraction if recover_edge_stars else 1.0,
    minSnr=min_snr,
    maxEllipticity=max_ellipticity,
    edgeMargin=edge_margin,
)
print('Config:', config)

starTracker = GuiderStarTracker(guiderData, config)
stars = starTracker.trackGuiderStars(refCatalog=None)

print(f'\nTracked catalog shape: {stars.shape}')
print('Detectors tracked:', sorted(stars["detector"].unique()) if not stars.empty else 'none')
stars.head()

<a id='schema'></a>
## Identify the Output Table

The `stars` table has **one row per (guide star, stamp)**. Each guide sensor
contributes one tracked star, so a row is a single star measured on one 50 ms stamp.
The columns fall into the groups below. The two ellipticity analyses draw on distinct
column groups:

- **Per-stamp ellipticity** &rarr; *shape* group (`e1`, `e2`, `ixx`, `iyy`, `ixy`, `fwhm`).
- **Image-motion ellipticity** &rarr; *offsets* group (`dx`, `dy`, `dalt`, `daz`) vs `elapsed_time`.

In [ ]:
# Documented meaning of every output column, grouped by role.
column_doc = {
    # --- identifiers & metadata ---
    'trackid':      ('id',       'Unique per-row id: detid*1000 + stamp'),
    'detector':     ('id',       'Guide sensor name, e.g. R44_SG1'),
    'detid':        ('id',       'Numeric detector id (guiderNameMap)'),
    'expid':        ('id',       'Exposure id = dayObs*100000 + seqNum'),
    'stamp':        ('id',       'Stamp index 0..nStamps-1 (one 50 ms integration)'),
    'ampname':      ('meta',     'Guider amplifier name'),
    'filter':       ('meta',     'Photometric band'),
    'exptime':      ('meta',     'Per-stamp integration time (s)'),
    'timestamp':    ('time',     'Absolute time of the stamp (astropy Time)'),
    'elapsed_time': ('time',     'Seconds since first stamp of reference guider'),

    # --- per-stamp shape (GalSim HSM adaptive moments) ---
    'fwhm':         ('shape',    'FWHM = 2.355*sigma, converted to arcsec'),
    'ixx':          ('shape',    'Second moment sigma^2*(1+e1) [pix^2]'),
    'iyy':          ('shape',    'Second moment sigma^2*(1-e1) [pix^2]'),
    'ixy':          ('shape',    'Second moment sigma^2*e2 [pix^2]'),
    'e1':           ('shape',    'Ellipticity e1 in detector frame'),
    'e2':           ('shape',    'Ellipticity e2 in detector frame'),
    'e1_altaz':     ('shape',    'e1 rotated to Alt/Az frame (camRotAngle)'),
    'e2_altaz':     ('shape',    'e2 rotated to Alt/Az frame (camRotAngle)'),

    # --- photometry ---
    'flux':         ('phot',     'GalSim moments amplitude / aperture flux'),
    'flux_err':     ('phot',     'Flux uncertainty'),
    'snr':          ('phot',     'Signal-to-noise ratio'),
    'magoffset':    ('phot',     'Mag offset vs per-detector median flux (mmag)'),

    # --- centroid / positions ---
    'xroi':         ('pos',      'Centroid x in ROI/amp pixel coords'),
    'yroi':         ('pos',      'Centroid y in ROI/amp pixel coords'),
    'xerr':         ('pos',      'Centroid x uncertainty (pix)'),
    'yerr':         ('pos',      'Centroid y uncertainty (pix)'),
    'xccd':         ('pos',      'Centroid x in CCD pixel coords'),
    'yccd':         ('pos',      'Centroid y in CCD pixel coords'),
    'xfp':          ('pos',      'Focal-plane x (mm)'),
    'yfp':          ('pos',      'Focal-plane y (mm)'),
    'alt':          ('pos',      'Altitude (deg)'),
    'az':           ('pos',      'Azimuth (deg)'),
    'theta':        ('pos',      'Rotator angle atan2(yfp,xfp) (deg)'),
    'theta_err':    ('pos',      'Rotator angle uncertainty (arcsec)'),

    # --- per-detector reference (median over stamps) ---
    'xroi_ref':     ('ref',      'Median xroi over stamps for this detector'),
    'yroi_ref':     ('ref',      'Median yroi over stamps'),
    'xccd_ref':     ('ref',      'Median xccd over stamps'),
    'yccd_ref':     ('ref',      'Median yccd over stamps'),
    'xfp_ref':      ('ref',      'Median xfp over stamps'),
    'yfp_ref':      ('ref',      'Median yfp over stamps'),
    'alt_ref':      ('ref',      'Median alt over stamps'),
    'az_ref':       ('ref',      'Median az over stamps'),
    'theta_ref':    ('ref',      'Median theta over stamps'),

    # --- per-stamp offsets from reference (IMAGE MOTION) ---
    'dx':           ('offset',   'xccd - xccd_ref (pix)'),
    'dy':           ('offset',   'yccd - yccd_ref (pix)'),
    'dxfp':         ('offset',   'xfp - xfp_ref (mm)'),
    'dyfp':         ('offset',   'yfp - yfp_ref (mm)'),
    'dalt':         ('offset',   '(alt - alt_ref) (arcsec)'),
    'daz':          ('offset',   '(az - az_ref)*cos(alt) (arcsec)'),
    'dtheta':       ('offset',   '(theta - theta_ref) (arcsec)'),
}

In [ ]:
# Build a tidy schema table: column, group, dtype, present?, description.
present = list(stars.columns)
rows = []
for col in present:
    group, desc = column_doc.get(col, ('?', 'UNDOCUMENTED'))
    rows.append((col, group, str(stars[col].dtype), desc))
schema = pd.DataFrame(rows, columns=['column', 'group', 'dtype', 'description'])

# Flag any documented columns missing from the actual output, and vice versa.
missing = [c for c in column_doc if c not in present]
extra   = [c for c in present if c not in column_doc]
if missing:
    print('Documented but NOT in output:', missing)
if extra:
    print('In output but UNDOCUMENTED:', extra)

print(f'\n{len(present)} columns, grouped:')
print(schema.groupby('group').size().to_string())
schema

In [ ]:
# Column groups as reusable lists for the downstream ellipticity analyses.
groups = {g: schema.loc[schema.group == g, 'column'].tolist()
          for g in schema.group.unique()}

shape_cols  = groups.get('shape', [])
offset_cols = groups.get('offset', [])

print('Per-stamp ellipticity columns :', shape_cols)
print('Image-motion (offset) columns :', offset_cols)

<a id='summary'></a>
## Per-Detector Summary

In [ ]:
# How many stamps were tracked per guide sensor, plus mean shape/SNR.
summary = (stars.groupby('detector')
                 .agg(n_stamps=('stamp', 'count'),
                      snr_med=('snr', 'median'),
                      fwhm_med=('fwhm', 'median'),
                      e1_med=('e1', 'median'),
                      e2_med=('e2', 'median'),
                      dx_rms=('dx', lambda s: np.nanstd(s)),
                      dy_rms=('dy', lambda s: np.nanstd(s)))
                 .round(4))
print(f'Expected stamps per detector ~ {len(guiderData)}')
summary

In [ ]:
# Full numeric summary of the shape and offset columns.
stars[shape_cols + offset_cols].describe().T

<a id='images'></a>
## Guide-star images (movie & coadd)

Zoomed-in views of the tracked guide stars, to assess the raw images alongside the tracker
output and the moment decomposition below. Uses `summit_utils`' `GuiderPlotter`.

- **Movie**: the per-stamp (50 ms) cutouts played in sequence &mdash; shows the image motion
  and shape jitter directly.
- **Coadd**: the stacked image (same zoom).

Note: `GuiderPlotter`/`getStampArrayCoadd` display the **median** stack, whereas the moment
additivity test builds its own **mean** coadd (a median partly rejects the centroid wander),
so the displayed coadd and the `coadd` row of the moment table differ slightly by design.

In [ ]:
from lsst.summit.utils.guiders.plotting import GuiderPlotter

# Pass stars so the plotter can zoom on each star center (cutoutSize > 0).
plotter = GuiderPlotter(guiderData, stars)

In [ ]:
# Static coadd mosaic, zoomed on the tracked star in each sensor.
fig = plotter.plotMosaic(
    stampNum=-1,               # -1 => stacked (coadd) image
    cutoutSize=image_cutout,
    plo=image_plo, phi=image_phi,
    saveAs=f'coadd_mosaic_{expid}.png',
)

In [ ]:
from IPython.display import Image, display

# Per-stamp movie (zoomed). Saved as a gif, then displayed inline.
gif = f'star_movie_{expid}.gif'
plotter.makeAnimation(
    cutoutSize=image_cutout,
    plo=image_plo, phi=image_phi,
    fps=int(guider_hz),
    saveAs=gif,
)
display(Image(filename=gif))

### RubinTV-style movies (full frame + star cutout)

The two animations RubinTV produces, with its exact parameters, for side-by-side comparison
with the analysis above (`GuiderPlotter.makeAnimation`, from `rubintv_production/guiders.py`):

- **full_movie**: `cutoutSize=-1` (whole ROI), `plo=70, phi=99`, `fps=5`.
- **star_movie**: `cutoutSize=20` (zoom on the tracked star), `plo=50, phi=98`, `fps=10`.

The stretch is a **per-panel, per-frame percentile** over the *displayed* region
(`np.nanpercentile(region, [plo, phi])`). So a panel whose full ROI contains a bright
contaminant (trail / saturated source) gets a high `vmax`, which washes a fainter guide star
to white in the full_movie even though it is tracked and shows fine in the star cutout. Writing
`.mp4` needs the ffmpeg writer (use `.gif` for the pillow writer).

In [ ]:
from IPython.display import Video

# full_movie: full ROI, RubinTV stretch (plo=70, phi=99), fps=5 (default)
full_fname = f'full_movie_{expid}.mp4'
plotter.makeAnimation(cutoutSize=-1, plo=70, phi=99, saveAs=full_fname)
Video(full_fname, embed=True, width=600)

In [ ]:
# star_movie: 20-px cutout on each tracked star, RubinTV stretch (plo=50, phi=98), fps=10
star_fname = f'star_movie_{expid}.mp4'
plotter.makeAnimation(cutoutSize=20, plo=50, phi=98, fps=10, saveAs=star_fname)
Video(star_fname, embed=True, width=600)

<a id='ellip'></a>
## Shape from Second Moments: Optics vs Turbulence

**Goal.** Separate the guide-star *shape* into a *static* part (bounds the optical
misalignment contribution) and a *time-variable* part (turbulence tip/tilt, Z2/Z3).
Everything below is in **unnormalized second moments** (pixel$^2$, reported in arcsec$^2$
via the pixel scale) &mdash; never the normalized ellipticity $e_i=Q_i/T$.

We use the moment quantities

$$M_{xx},\;M_{yy},\;M_{xy}\quad\Rightarrow\quad
Q_1\equiv M_{xx}-M_{yy},\;\; Q_2\equiv 2M_{xy},\;\; T\equiv M_{xx}+M_{yy}.$$

$Q_1,Q_2$ carry the shape (traceless part) and $T$ the size. Unlike $e_i$, these **add
linearly**. For a **mean** coadd of stamps in a common pixel frame the law of total
variance is exact:

$$M^{\rm coadd}_{ab}=\langle M^{\rm stamp}_{ab}\rangle+{\rm Cov}(x_c,y_c)_{ab}$$

i.e. $Q_i^{\rm co}=\langle Q_i\rangle+Q_i^{\rm mot}$ and $T^{\rm co}=\langle T\rangle+T^{\rm mot}$, with the
image-motion terms from the per-stamp centroid covariance
$Q_1^{\rm mot}={\rm Var}(x)-{\rm Var}(y)$, $Q_2^{\rm mot}=2\,{\rm Cov}(x,y)$, $T^{\rm mot}={\rm Var}(x)+{\rm Var}(y)$.

**Interpretation (working assumption).**
- $\langle Q_i\rangle_{\rm stamp}$ = static shape moment &rarr; **upper bound on the optics contribution**
  (optics + residual within-stamp turbulence anisotropy, the latter averaging down as $1/\sqrt{N}$).
- $Q_i^{\rm mot}$ = **turbulence tip/tilt** contribution (the $M_{xx}-M_{yy}$, $2M_{xy}$ of the
  per-stamp centroid distribution) &mdash; exactly the comparison requested.

**Two caveats handled in the code:**
1. `getStampArrayCoadd` is a *median* stack &mdash; the identity needs a *mean* stack, so we
   build our own mean coadd (raw ROI frame, no re-registration).
2. The tracker's `ixx/iyy/ixy` come from HSM *adaptive* (Gaussian-weighted) moments, which
   break the identity. We recompute *unweighted, fixed-aperture* moments consistently across
   stamps, coadd, and centroids so that $Q_i$ and $T$ are directly additive.

### Moment measurement (from `guiderMoments`)

The moment-measurement primitives and the optics/turbulence decomposition now live in the
reusable module `code/guiderMoments.py` (imported above), so the notebook and the Snakemake
pipeline share one implementation. The scheme (justified in `moment_weighting_analysis.md`):

- **Centroid**: fixed-Gaussian *weighted* centroid (`gaussianWeightedCentroid`).
- **Per-stamp shape**: *unweighted* moments about that centroid (`unweightedMoments`).
- **Image motion**: flux-weighted covariance of the per-stamp weighted centroids.
- **Noise floor**: `<err**2>` subtracted from the mean per-stamp shape and the motion.

`decomposeExposure` runs this per sensor and returns `DetectorMoments` (arcsec$^2$);
`momentsToDataFrame` flattens them to the tidy table used below and by the pipeline.

In [ ]:
import guiderMoments as gm
print('guiderMoments exports:')
print('  ' + ', '.join(gm.__all__))

### A. Do the second moments add up?  (coadd vs mean-stamp + motion)

Per guide sensor, in arcsec$^2$, using the weighted-centroid scheme above:
- `coadd`      : moments of the **mean** coadd (np.nanmean stack), centered on the mean of
  the per-stamp weighted centroids $\bar{\tilde c}$,
- `stamp_mean` : flux-weighted mean of the per-stamp unweighted moments (about $\tilde c_i$),
  **noise-floor subtracted**,
- `motion`     : flux-weighted covariance of the per-stamp weighted centroids,
  **noise-floor subtracted** ($M_{xx}-M_{yy}$, $2M_{xy}$ of the centroid distribution).

The aperture radius is $4\sigma$ from the per-detector median HSM FWHM (e.g. FWHM $\approx1''$
at $0.2''$/pix $\Rightarrow \sigma\approx2.1$ pix $\Rightarrow r_{\rm ap}\approx8.5$ pix). Test
$M^{\rm coadd}\approx M^{\rm stamp\_mean}+M^{\rm motion}$ term by term. Motion terms may go
slightly negative when the true motion is below the centroid-noise floor.

In [ ]:
# Decompose every tracked sensor with the shared library.
momentConfig = MomentConfig(apNSigma=ap_nsigma, cenNIter=cen_niter)
decomps = decomposeExposure(guiderData, stars, momentConfig)

mom = momentsToDataFrame(decomps)
per_stamp_moments = {det: dm.perStamp for det, dm in decomps.items()}
noise_floor = {det: dm.noiseFloor for det, dm in decomps.items()}

nf = pd.DataFrame([(d, nx, ny) for d, (nx, ny) in noise_floor.items()],
                  columns=['detector', 'Nx_arcsec2', 'Ny_arcsec2']).round(5)
print(f'{len(decomps)} sensors processed. Moments in arcsec^2.')
print('Centroid-noise floor (arcsec^2), subtracted from stamp_mean & motion:')
print(nf.to_string(index=False))
mom.round(5)

In [ ]:
# Residual of the additivity identity per sensor: coadd - (stamp_mean + motion), arcsec^2
piv = mom.pivot_table(index='detector', columns='kind', values=['Q1', 'Q2', 'T'])
resid = pd.DataFrame(index=piv.index)
for q in ['Q1', 'Q2', 'T']:
    resid[f'{q}_coadd'] = piv[(q, 'coadd')]
    resid[f'{q}_s+m']   = piv[(q, 'stamp+motion')]
    resid[f'{q}_resid'] = piv[(q, 'coadd')] - piv[(q, 'stamp+motion')]
    resid[f'{q}_frac']  = resid[f'{q}_resid'] / piv[(q, 'coadd')]
print('Additivity residuals: coadd - (stamp_mean + motion).  frac = resid/coadd')
resid.round(5)

### B. Static shape moment (optics bound) vs image-motion moment (turbulence)

The requested comparison, all in arcsec$^2$:
- **Optics upper bound** = $\langle Q_1\rangle,\langle Q_2\rangle$ (flux-weighted mean per-stamp shape moment),
- **Turbulence (image motion)** = $Q_1^{\rm mot}=M_{xx}-M_{yy}$, $Q_2^{\rm mot}=2M_{xy}$ of the centroids.

Also report the size moments $T$ so the relative amplitude of each contribution is explicit.

In [ ]:
split = []
for det, g in mom.groupby('detector'):
    gg = g.set_index('kind')
    split.append(dict(
        detector=det, xfp=gg.loc['coadd', 'xfp'], yfp=gg.loc['coadd', 'yfp'],
        # optics bound (static per-stamp shape moment)
        Q1_optics=gg.loc['stamp_mean', 'Q1'], Q2_optics=gg.loc['stamp_mean', 'Q2'],
        # turbulence image-motion shape moment
        Q1_motion=gg.loc['motion', 'Q1'], Q2_motion=gg.loc['motion', 'Q2'],
        # coadd (total) shape moment
        Q1_coadd=gg.loc['coadd', 'Q1'], Q2_coadd=gg.loc['coadd', 'Q2'],
        # size moments
        T_stamp=gg.loc['stamp_mean', 'T'], T_motion=gg.loc['motion', 'T'],
        T_coadd=gg.loc['coadd', 'T'],
    ))
split = pd.DataFrame(split)
print('All quantities in arcsec^2. Q1 = Mxx-Myy, Q2 = 2*Mxy, T = Mxx+Myy.')
split.round(5)

### Additivity scatter: coadd vs (stamp_mean + motion)

One point per guide sensor. If the second moments add ($M^{\rm co}=\langle M\rangle+M^{\rm mot}$)
the points lie on the dashed $y=x$ line, for each of $Q_1$, $Q_2$ (shape) and $T$ (size).

In [ ]:
panels = [
    ('Q1_optics', 'Q1_motion', 'Q1_coadd', 'Q1 = Mxx-Myy'),
    ('Q2_optics', 'Q2_motion', 'Q2_coadd', 'Q2 = 2Mxy'),
    ('T_stamp',   'T_motion',  'T_coadd',  'T = Mxx+Myy'),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (oc, mc, cc, lab) in zip(axes, panels):
    x = (split[oc] + split[mc]).to_numpy()
    y = split[cc].to_numpy()
    ax.scatter(x, y, c=range(len(split)), cmap='viridis', s=60, edgecolor='k', zorder=3)
    lo = float(np.nanmin([x.min(), y.min()])); hi = float(np.nanmax([x.max(), y.max()]))
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], 'k--', lw=1, label='y = x')
    ax.set_xlabel(f'{lab}: stamp_mean + motion [arcsec$^2$]')
    ax.set_ylabel(f'{lab}: coadd [arcsec$^2$]')
    ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=9)
fig.suptitle(f'Additivity: coadd vs (stamp_mean + motion)   {expid}')
fig.tight_layout()

### Static (optics) fraction of the shape moment

Histogram over sensors of $Q_i^{\rm stamp}/(Q_i^{\rm stamp}+Q_i^{\rm mot})$ &mdash; the fraction of
the combined shape moment that is static (optics-bound) vs from image motion (turbulence).
Because $Q_1,Q_2$ are **signed**, the ratio is not bounded to $[0,1]$ when the static and
motion terms have opposite signs; markers at 0 and 1 delimit the 'clean' range. With only
~8 sensors per exposure this is thin &mdash; it becomes meaningful aggregated over many
exposures (Snakemake pipeline).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (oc, mc, lab) in zip(axes, [('Q1_optics', 'Q1_motion', 'Q1'),
                                    ('Q2_optics', 'Q2_motion', 'Q2')]):
    denom = split[oc] + split[mc]
    frac = (split[oc] / denom).replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(frac, bins=np.linspace(-0.5, 1.5, 21), color='C0', edgecolor='k', alpha=0.8)
    ax.axvline(0.0, color='k', ls=':', lw=1)
    ax.axvline(1.0, color='k', ls=':', lw=1)
    ax.axvline(0.5, color='grey', ls='--')
    ax.set_xlabel(f'{lab}(stamp_mean) / [{lab}(stamp_mean) + {lab}(motion)]')
    ax.set_ylabel('detectors')
    ax.set_title(f'{lab}: static (optics) fraction')
fig.suptitle(f'Static vs motion fraction of the shape moment   {expid}')
fig.tight_layout()

### Focal-plane whisker plot (moment space)

Whiskers show the traceless shape moment vector $(Q_1,Q_2)=(M_{xx}-M_{yy},\,2M_{xy})$ in
arcsec$^2$: length $\propto\sqrt{Q_1^2+Q_2^2}$, orientation $\tfrac12\arctan2(Q_2,Q_1)$.
Left = static per-stamp (optics bound); right = image motion (turbulence). A smooth
low-order pattern on the left indicates field-dependent optical aberrations; a coherent
orientation on the right indicates common tip/tilt. Both panels share one scale.

In [ ]:
def whisker(ax, x, y, Q1, Q2, scale, **kw):
    mag = np.hypot(Q1, Q2)
    ang = 0.5 * np.arctan2(Q2, Q1)
    dx = scale * mag * np.cos(ang)
    dy = scale * mag * np.sin(ang)
    for xi, yi, dxi, dyi in zip(x, y, dx, dy):
        ax.plot([xi - dxi, xi + dxi], [yi - dyi, yi + dyi], **kw)

# common scale from the larger of the two moment sets
mmax = np.nanmax(np.hypot(
    np.r_[split['Q1_optics'], split['Q1_motion']],
    np.r_[split['Q2_optics'], split['Q2_motion']]))
span = np.nanmax(np.hypot(split['xfp'] - split['xfp'].mean(),
                          split['yfp'] - split['yfp'].mean()))
scale = 0.15 * span / mmax if mmax > 0 else 1.0

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
for ax, (q1, q2, c, ttl) in zip(axes, [
        ('Q1_optics', 'Q2_optics', 'C0', 'Static per-stamp  <Mxx-Myy>, <2Mxy>  (optics bound)'),
        ('Q1_motion', 'Q2_motion', 'C3', 'Image motion  Var(x)-Var(y), 2Cov(x,y)  (turbulence)')]):
    whisker(ax, split['xfp'], split['yfp'], split[q1], split[q2], scale, color=c, lw=2)
    ax.scatter(split['xfp'], split['yfp'], s=10, color='k', zorder=3)
    ax.set_title(ttl, fontsize=10); ax.set_xlabel('xfp'); ax.set_aspect('equal'); ax.grid(alpha=0.3)
axes[0].set_ylabel('yfp')
fig.suptitle(f'Guider shape moments  {expid}   (arcsec^2, common whisker scale)')
fig.tight_layout()

### Temporal PSD: expected shape and the variance connection

Under **Taylor frozen flow** a phase screen blows across the aperture at wind speed $v$, locking
temporal to spatial frequency, $f=v\kappa$. Image motion is the aperture-averaged wavefront
gradient (Zernike tip/tilt, Z2/Z3), so the centroid PSD has:

- a **low-frequency plateau** (structure larger than the aperture &rarr; slow correlated wander),
- a **knee** at $f_c\approx0.3\,v/D$ (for Rubin $D=8.4$ m, $v\sim5$–$20$ m/s $\Rightarrow f_c\approx0.2$–$0.7$ Hz),
- a **steep rolloff** ($\sim f^{-11/3}$, steepening to $f^{-17/3}$ for aperture-filtered Zernike
  modes; Conan, Rousset & Madec 1995). The tip/tilt bandwidth scale is the *Tyler frequency*
  $f_T\propto[\int C_n^2 v^2 dh]^{1/2}$; the *Greenwood frequency* $f_G\approx0.43\,v/r_0$ instead sets
  the high-order phase bandwidth.

**Variance = integral of the PSD** (Wiener&ndash;Khinchin / Parseval): for zero-mean stationary $x(t)$,
$\mathrm{Var}(x)=\int_0^\infty P_x(f)\,df$. So the motion moment equals the area under the centroid
PSD, $M^{\rm mot}_{xx}=\mathrm{Var}(x_c)=\int P_{x_c}df$ &mdash; an internal consistency check &mdash; and the PSD
lets us attribute that variance to timescales: DC (removed) = static/optics; sub-knee = slow
drift / mount / wind shake; above the knee = atmospheric tip/tilt.

**Sampling caveats at 5 Hz.** Nyquist = 2.5 Hz (power above aliases back, modest given the steep
rolloff). The 50 ms integration is a boxcar &rarr; PSD $\times\,\mathrm{sinc}^2(\pi f\tau)$, $\tau=50$ ms
(first null 20 Hz; $\sim$5% at Nyquist). Physically, tilt faster than $\sim$20 Hz is averaged into
per-stamp *blur* (adds to the shape/size moment) rather than into centroid *motion* &mdash; the
boundary between our two channels.

### Per-stamp moment & centroid time series + PSD (one sensor)

Temporal mean of $Q_1,Q_2$ = static/optics-like part; fluctuation = turbulence. The centroid PSD
is the tip/tilt spectrum; its integral equals the motion moment (Parseval check printed). Sensor
with the most stamps.

In [ ]:
det0 = max(per_stamp_moments, key=lambda d: len(per_stamp_moments[d]))
pm = per_stamp_moments[det0].sort_values('stamp').reset_index(drop=True)  # arcsec^2
pixscale = pm['pixscale'].iloc[0]
t = pm['stamp'].to_numpy() / guider_hz  # seconds

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
ax[0, 0].plot(t, pm['Q1'], '.-', label='Q1 = Mxx-Myy')
ax[0, 0].plot(t, pm['Q2'], '.-', label='Q2 = 2Mxy')
ax[0, 0].axhline(pm['Q1'].mean(), color='C0', ls='--')
ax[0, 0].axhline(pm['Q2'].mean(), color='C1', ls='--')
ax[0, 0].set_title(f'{det0}: per-stamp shape moment [arcsec^2]')
ax[0, 0].set_xlabel('t [s]'); ax[0, 0].legend()

ax[0, 1].plot(t, pm['T'], '.-', color='C2', label='T = Mxx+Myy')
ax[0, 1].axhline(pm['T'].mean(), color='C2', ls='--')
ax[0, 1].set_title(f'{det0}: per-stamp size moment [arcsec^2]')
ax[0, 1].set_xlabel('t [s]'); ax[0, 1].legend()

# centroid motion in arcsec (weighted centroids) and its PSD (Parseval check)
xb = (pm['xc'] - pm['xc'].mean()) * pixscale
yb = (pm['yc'] - pm['yc'].mean()) * pixscale
psd_last = None
for series, lab, col in [(xb.to_numpy(), 'dx', 'C0'), (yb.to_numpy(), 'dy', 'C1')]:
    freq, psd, var = centroidPsd(series, guider_hz)
    ax[1, 0].loglog(freq[1:], psd[1:], '.-', color=col, label=lab)
    psd_last = (freq, psd)
    print(f'{det0} {lab}: Var={series.var():.4g}, int(PSD)df={var:.4g} arcsec^2')
freq, psd = psd_last
fref = freq[1:]
ax[1, 0].loglog(fref, psd[1] * (fref / fref[0]) ** (-11.0 / 3), 'k:', lw=1, label='f^-11/3')
ax[1, 0].set_title('centroid PSD [arcsec^2/Hz]'); ax[1, 0].set_xlabel('f [Hz]'); ax[1, 0].legend()

ax[1, 1].scatter(xb, yb, s=8, c=t, cmap='viridis')
ax[1, 1].set_title('centroid scatter [arcsec] (color=t)')
ax[1, 1].set_xlabel('dx'); ax[1, 1].set_ylabel('dy'); ax[1, 1].set_aspect('equal'); ax[1, 1].grid(alpha=0.3)
fig.tight_layout()

### Notes & next steps

- **Common-mode motion across sensors** (turbulence tip/tilt vs differential): stack the
  per-sensor centroid series (aligned by `timestamp`) and separate the field-average global
  tip/tilt / mount motion from the residual, all as motion moments $M_{xx}-M_{yy}$, $2M_{xy}$.
- **Field model for optics**: fit the static $\langle Q_1\rangle,\langle Q_2\rangle(x_{\rm fp},y_{\rm fp})$ to a
  low-order 2D / Zernike-gradient basis once more sensors/exposures are available.
- **Seeing / airmass regression** across exposures: intercept &rarr; optics moment, slope &rarr; turbulence.
- **Camera vs Alt/Az**: repeat with moments rotated to the Alt/Az frame to isolate sky-fixed
  effects (DCR elongation toward zenith).
- **GuiderStarTracker improvements this motivates**: (1) a mean-coadd option; (2) consistent
  fixed-weight/unweighted second moments (Mxx,Myy,Mxy) in the output alongside HSM adaptive;
  (3) track multiple stars per sensor; (4) expose per-stamp raw moments and centroid covariance.